In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler
from prophet import Prophet
from xgboost import XGBRegressor


from lightgbm import LGBMRegressor
from xgboost import XGBRegressor


In [45]:
DATA_PATH = "./csv_export/main_dataset.csv"
TARGET = "NbPaxTotal"

TRAIN_END = pd.Timestamp("2026-02-28 23:59:59")
TEST_END  = pd.Timestamp("2026-03-22 23:59:59")

In [46]:
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

(364745, 231)


,IdMovementVinci,IdFarms,IdMovement,IdADL,IdPkgStand,IdTraficType,IdIrregularityCode,IdRunway,IdAircraftType,IdBusinessUnitType,...,is_fr_school_holiday_zone_c,is_first_last_day_of_school_holiday,is_origin_public_holiday,is_origin_school_holiday,is_dest_public_holiday,is_dest_school_holiday,scheduled_date,is_any_day_off,days_until_next_day_off,days_until_next_workday
0,ACA876|A|2025-04-14 08:35:00+00,1.016947e+09,20250414080700CGHKW,20250414103500ACA00876,D,1,NaN,17L,333,1,...,1,0,0,0,0,0,2025-04-14,0,5,0.0
1,ACA876|A|2024-05-27 07:55:00+00,1.015952e+09,20240527072400CGHPX,20240527095500ACA00876,C,1,NaN,35R,788,1,...,0,0,0,0,0,0,2024-05-27,0,5,0.0
2,ACA876|A|2024-01-14 10:00:00+00,1.015522e+09,20240114103200CGFAF,20240114110000ACA00876,D,1,NaN,17L,333,1,...,0,0,0,0,0,0,2024-01-14,1,0,1.0
3,ACA876|A|2024-05-11 07:55:00+00,1.015909e+09,20240511070900CGHPX,20240511095500ACA00876,C,1,NaN,17L,788,1,...,0,0,0,0,0,0,2024-05-11,1,0,2.0
4,AF1489|A|2025-10-03 17:45:00+00,1.017600e+09,20251003191200FHBLC,20251003194500AFR01489,L,1,RTD,17L,E90,1,...,0,0,0,0,0,0,2025-10-03,0,1,0.0


In [47]:
cols = [
    "IdAircraftType",
    "IdBusinessUnitType",
    "IdBusContactType",
    "airlineOACICode",
    "AirportPrevious",
    "ServiceCode",
    "FlightNumberNormalized",
    "LTScheduledDatetime",
    "Direction",
    "SysTerminal",
    "NbOfSeats",

    #Calendar and route features generated in update_datasets.py
    "day_of_week",
    "is_weekend",
    "season",
    "origin_country",
    "dest_country",
    "is_domestic",
    "is_route_domestic",
    "has_stopover",

    # Holiday features generated in update_datasets.py
    "is_fr_public_holiday",
    "is_bridge_day",
    "is_fr_school_holiday_zone_a",
    "is_fr_school_holiday_zone_b",
    "is_fr_school_holiday_zone_c",
    "is_first_last_day_of_school_holiday",
    "is_origin_public_holiday",
    "is_origin_school_holiday",
    "is_dest_public_holiday",
    "is_dest_school_holiday",

    # Weather features generated in update_datasets.py
    "precipitation_sum",
    "rain_sum",
    "snowfall_sum",
    "windspeed_10m_max",

    # Engineered datetime and countdown features
    "month_of_year",
    "hour_of_day",
    "is_any_day_off",
    "days_until_next_day_off",
    "days_until_next_workday",

    TARGET
]

df = df[cols].copy()
print(df.shape)

(364745, 39)


In [48]:
df["LTScheduledDatetime"] = pd.to_datetime(df["LTScheduledDatetime"], errors="coerce")

numeric_cols = [
    "is_weekend",
    "IdAircraftType",
    "IdBusinessUnitType",
    "IdBusContactType",
    "NbOfSeats",
    "precipitation_sum",
    "rain_sum",
    "snowfall_sum",
    "windspeed_10m_max",
    
    "days_until_next_day_off",
    "days_until_next_workday",
    TARGET
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

binary_cols = [
    "has_stopover",
    "is_domestic",
    "is_route_domestic",
    "is_fr_public_holiday",
    "is_bridge_day",
    "is_fr_school_holiday_zone_a",
    "is_fr_school_holiday_zone_b",
    "is_fr_school_holiday_zone_c",
    "is_first_last_day_of_school_holiday",
    "is_origin_public_holiday",
    "is_origin_school_holiday",
    "is_dest_public_holiday",
    "is_dest_school_holiday",
]

for col in binary_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [50]:
# On enlève le trafic type 0
#df = df[df["IdTraficType"] != 0].copy()

# On coupe au 22/03/2026
df = df[df["LTScheduledDatetime"] <= TEST_END].copy()

# On garde seulement les lignes où NbOfSeats est exploitable
df = df[df["NbOfSeats"].notna()].copy()
df = df[df["NbOfSeats"] > 0].copy()

print(df.shape)
print(df["LTScheduledDatetime"].min(), "->", df["LTScheduledDatetime"].max())
print("NaN NbOfSeats:", df["NbOfSeats"].isna().sum())

(303360, 39)
2023-01-01 00:05:00 -> 2026-03-22 23:55:00
NaN NbOfSeats: 0


In [51]:
df_known = df[df[TARGET].notna()].copy()
df_known["date"] = pd.to_datetime(df_known["LTScheduledDatetime"]).dt.floor("D")

print(df_known.shape)
df_known.head()

(303360, 40)


,IdAircraftType,IdBusinessUnitType,IdBusContactType,airlineOACICode,AirportPrevious,ServiceCode,FlightNumberNormalized,LTScheduledDatetime,Direction,SysTerminal,...,rain_sum,snowfall_sum,windspeed_10m_max,month_of_year,hour_of_day,is_any_day_off,days_until_next_day_off,days_until_next_workday,NbPaxTotal,date
0,333.0,1,1,ACA,YUL,J,ACA00876,2025-04-14 10:35:00,Arrivée,T1,...,0.1,0.0,13.8,4,10.583333,0,5,0.0,222.0,2025-04-14
1,788.0,1,1,ACA,YUL,J,ACA00876,2024-05-27 09:55:00,Arrivée,T1,...,4.5,0.0,12.7,5,9.916667,0,5,0.0,216.0,2024-05-27
2,333.0,1,1,ACA,YUL,J,ACA00876,2024-01-14 11:00:00,Arrivée,T1,...,0.0,0.0,7.0,1,11.000000,1,0,1.0,270.0,2024-01-14
3,788.0,1,1,ACA,YUL,J,ACA00876,2024-05-11 09:55:00,Arrivée,T1,...,0.0,0.0,15.5,5,9.916667,1,0,2.0,255.0,2024-05-11
4,NaN,1,3,AFR,NTE,J,AFR01489,2025-10-03 19:45:00,Arrivée,T1,...,0.0,0.0,4.8,10,19.750000,0,1,0.0,101.0,2025-10-03


In [52]:
train_df = df_known[df_known["LTScheduledDatetime"] <= TRAIN_END].copy()

test_df = df_known[
    (df_known["LTScheduledDatetime"] > TRAIN_END) &
    (df_known["LTScheduledDatetime"] <= TEST_END)
].copy()

for d in [train_df, test_df]:
    d["date"] = pd.to_datetime(d["LTScheduledDatetime"]).dt.floor("D")

print("train:", train_df.shape)
print("test :", test_df.shape)

train: (297687, 40)
test : (5673, 40)


In [53]:
daily_all = (
    df_known.groupby("date", as_index=False)
    .agg(
        daily_pax=(TARGET, "sum"),
        daily_seats=("NbOfSeats", "sum"),
        day_of_week=("day_of_week", "first"),
        is_weekend=("is_weekend", "max"),
        is_fr_public_holiday=("is_fr_public_holiday", "max"),
        is_fr_school_holiday_zone_a=("is_fr_school_holiday_zone_a", "max"),
        is_any_day_off=("is_any_day_off", "max"),
        precipitation_sum=("precipitation_sum", "sum"),
        rain_sum=("rain_sum", "sum"),
        snowfall_sum=("snowfall_sum", "sum"),
        windspeed_10m_max=("windspeed_10m_max", "mean"),
        days_until_next_day_off=("days_until_next_day_off", "mean"),
        days_until_next_workday=("days_until_next_workday", "mean"),
    )
    .sort_values("date")
    .reset_index(drop=True)
)

daily_all["daily_load_factor_true"] = daily_all["daily_pax"] / np.clip(daily_all["daily_seats"], 1, None)

daily_all.head()

,date,daily_pax,daily_seats,day_of_week,is_weekend,is_fr_public_holiday,is_fr_school_holiday_zone_a,is_any_day_off,precipitation_sum,rain_sum,snowfall_sum,windspeed_10m_max,days_until_next_day_off,days_until_next_workday,daily_load_factor_true
0,2023-01-01,28276.0,35539.0,6,1,1,1,1,0.0,0.0,0.0,35.3,0.0,2.0,0.795633
1,2023-01-02,30356.0,38547.0,0,0,0,1,1,3299.4,3299.4,0.0,40.0,0.0,1.0,0.787506
2,2023-01-03,23855.0,33528.0,1,0,0,0,0,701.8,701.8,0.0,10.5,4.0,0.0,0.711495
3,2023-01-04,19838.0,29939.0,2,0,0,0,0,0.0,0.0,0.0,7.4,3.0,0.0,0.662614
4,2023-01-05,19720.0,31878.0,3,0,0,0,0,302.4,302.4,0.0,7.5,2.0,0.0,0.618608


In [54]:
daily_train = daily_all[daily_all["date"] <= TRAIN_END.floor("D")].copy()

daily_test = daily_all[
    (daily_all["date"] > TRAIN_END.floor("D")) &
    (daily_all["date"] <= TEST_END.floor("D"))
].copy()

print(daily_train.shape, daily_test.shape)

(1155, 15) (22, 15)


In [55]:
prophet_train = daily_train[["date", "daily_pax"]].rename(
    columns={"date": "ds", "daily_pax": "y"}
).copy()

m_prophet = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False
)

m_prophet.fit(prophet_train)

11:58:45 - cmdstanpy - INFO - Chain [1] start processing
11:58:45 - cmdstanpy - INFO - Chain [1] done processing


In [56]:
future_dates = pd.DataFrame({
    "ds": daily_all["date"]
})

forecast_prophet = m_prophet.predict(future_dates)[["ds", "yhat"]].rename(
    columns={"ds": "date", "yhat": "daily_forecast_pax"}
)

forecast_prophet["daily_forecast_pax"] = np.clip(forecast_prophet["daily_forecast_pax"], 0, None)

daily_features = daily_all.merge(forecast_prophet, on="date", how="left")
daily_features["daily_forecast_load_factor"] = (
    daily_features["daily_forecast_pax"] / np.clip(daily_features["daily_seats"], 1, None)
)

daily_features["date"] = pd.to_datetime(daily_features["date"]).dt.floor("D")
daily_features.head()

,date,daily_pax,daily_seats,day_of_week,is_weekend,is_fr_public_holiday,is_fr_school_holiday_zone_a,is_any_day_off,precipitation_sum,rain_sum,snowfall_sum,windspeed_10m_max,days_until_next_day_off,days_until_next_workday,daily_load_factor_true,daily_forecast_pax,daily_forecast_load_factor
0,2023-01-01,28276.0,35539.0,6,1,1,1,1,0.0,0.0,0.0,35.3,0.0,2.0,0.795633,30357.973661,0.854216
1,2023-01-02,30356.0,38547.0,0,0,0,1,1,3299.4,3299.4,0.0,40.0,0.0,1.0,0.787506,27399.602850,0.710810
2,2023-01-03,23855.0,33528.0,1,0,0,0,0,701.8,701.8,0.0,10.5,4.0,0.0,0.711495,21818.222061,0.650746
3,2023-01-04,19838.0,29939.0,2,0,0,0,0,0.0,0.0,0.0,7.4,3.0,0.0,0.662614,21843.428072,0.729598
4,2023-01-05,19720.0,31878.0,3,0,0,0,0,302.4,302.4,0.0,7.5,2.0,0.0,0.618608,24444.155889,0.766803


In [57]:
daily_merge_cols = [
    "date",
    "daily_pax",
    "daily_seats",
    "daily_load_factor_true",
    "daily_forecast_pax",
    "daily_forecast_load_factor"
]

train_df = train_df.merge(daily_features[daily_merge_cols], on="date", how="left")
test_df = test_df.merge(daily_features[daily_merge_cols], on="date", how="left")

print(train_df.shape, test_df.shape)

(297687, 45) (5673, 45)


In [58]:
for d in [train_df, test_df]:
    d["target_load_factor_flight"] = d[TARGET] / np.clip(d["NbOfSeats"], 1, None)
    d["target_load_factor_flight"] = np.clip(d["target_load_factor_flight"], 0, 1.5)

train_df[["NbOfSeats", TARGET, "target_load_factor_flight"]].head()

,NbOfSeats,NbPaxTotal,target_load_factor_flight
0,297.0,222.0,0.747475
1,250.0,216.0,0.864000
2,297.0,270.0,0.909091
3,250.0,255.0,1.020000
4,100.0,101.0,1.010000


In [59]:
def add_group_stat(train_base, df_to_enrich, group_cols, target_col, stat_name, agg="mean", fill_value=None):
    df_to_enrich = df_to_enrich.copy()

    if stat_name in df_to_enrich.columns:
        df_to_enrich = df_to_enrich.drop(columns=[stat_name])

    stats = (
        train_base.groupby(group_cols, dropna=False)[target_col]
        .agg(agg)
        .reset_index()
        .rename(columns={target_col: stat_name})
    )

    out = df_to_enrich.merge(stats, on=group_cols, how="left")

    if fill_value is not None:
        out[stat_name] = out[stat_name].fillna(fill_value)

    return out

In [60]:
global_mean_lf = train_df["target_load_factor_flight"].mean()

group_features = [
    (["airlineOACICode"], "mean_lf_by_airline"),
    (["AirportPrevious"], "mean_lf_by_airport_previous"),
    (["ServiceCode"], "mean_lf_by_service"),
    (["SysTerminal"], "mean_lf_by_terminal"),
    (["day_of_week"], "mean_lf_by_dayofweek"),
    (["season"], "mean_lf_by_season"),
    (["IdAircraftType"], "mean_lf_by_aircraft"),
    (["airlineOACICode", "day_of_week"], "mean_lf_by_airline_day"),
    (["AirportPrevious", "day_of_week"], "mean_lf_by_airport_day"),
]

for cols_, feat_name in group_features:
    train_df = add_group_stat(train_df, train_df, cols_, "target_load_factor_flight", feat_name, agg="mean", fill_value=global_mean_lf)
    test_df = add_group_stat(train_df, test_df, cols_, "target_load_factor_flight", feat_name, agg="mean", fill_value=global_mean_lf)

In [61]:
for d in [train_df, test_df]:
    d["hour"] = d["LTScheduledDatetime"].dt.hour
    d["month"] = d["LTScheduledDatetime"].dt.month
    d["weekofyear"] = d["LTScheduledDatetime"].dt.isocalendar().week.astype(int)

    d["hour_sin"] = np.sin(2 * np.pi * d["hour"] / 24)
    d["hour_cos"] = np.cos(2 * np.pi * d["hour"] / 24)
    d["month_sin"] = np.sin(2 * np.pi * d["month"] / 12)
    d["month_cos"] = np.cos(2 * np.pi * d["month"] / 12)

    d["relative_seat_share_day"] = d["NbOfSeats"] / np.clip(d["daily_seats"], 1, None)
    d["forecast_pax_per_seat_day"] = d["daily_forecast_pax"] / np.clip(d["daily_seats"], 1, None)

In [62]:
categorical_cols = [
    "IdAircraftType",
    "IdBusinessUnitType",
    "IdBusContactType",
    "airlineOACICode",
    "AirportPrevious",
    "ServiceCode",
    "FlightNumberNormalized",
    "Direction",
    "SysTerminal",
    "season",
    "origin_country",
    "dest_country"
]

for col in categorical_cols:
    train_df[col] = train_df[col].astype(str)
    test_df[col] = test_df[col].astype(str)

    all_values = pd.concat([train_df[col], test_df[col]], axis=0).unique()
    mapping = {v: i for i, v in enumerate(all_values)}

    train_df[col] = train_df[col].map(mapping).astype(int)
    test_df[col] = test_df[col].map(mapping).astype(int)

In [63]:
feature_cols = [
    "IdAircraftType",
    "IdBusinessUnitType",
    "IdBusContactType",
    "airlineOACICode",
    "AirportPrevious",
    "ServiceCode",
    "FlightNumberNormalized",
    "Direction",
    "SysTerminal",
    "NbOfSeats",
    "day_of_week",
    "is_weekend",
    "season",
    "dest_country",
    "is_fr_public_holiday",
    "is_fr_school_holiday_zone_a",
    "is_dest_public_holiday",
    "is_dest_school_holiday",
    "precipitation_sum",
    "rain_sum",
    "snowfall_sum",
    "windspeed_10m_max",
    "is_any_day_off",
    "days_until_next_day_off",
    "days_until_next_workday",
    "hour",
    "month",
    "weekofyear",
    "hour_sin",
    "hour_cos",
    "month_sin",
    "month_cos",
    "daily_seats",
    "daily_forecast_pax",
    "daily_forecast_load_factor",
    "relative_seat_share_day",
    "forecast_pax_per_seat_day",
    "mean_lf_by_airline",
    "mean_lf_by_airport_previous",
    "mean_lf_by_service",
    "mean_lf_by_terminal",
    "mean_lf_by_dayofweek",
    "mean_lf_by_season",
    "mean_lf_by_aircraft",
    "mean_lf_by_airline_day",
    "mean_lf_by_airport_day",
]

target_col = "target_load_factor_flight"

print("Nombre de features:", len(feature_cols))

Nombre de features: 46


In [64]:
df

,IdAircraftType,IdBusinessUnitType,IdBusContactType,airlineOACICode,AirportPrevious,ServiceCode,FlightNumberNormalized,LTScheduledDatetime,Direction,SysTerminal,...,precipitation_sum,rain_sum,snowfall_sum,windspeed_10m_max,month_of_year,hour_of_day,is_any_day_off,days_until_next_day_off,days_until_next_workday,NbPaxTotal
0,333.0,1,1,ACA,YUL,J,ACA00876,2025-04-14 10:35:00,Arrivée,T1,...,0.1,0.1,0.00,13.8,4,10.583333,0,5,0.0,222.0
1,788.0,1,1,ACA,YUL,J,ACA00876,2024-05-27 09:55:00,Arrivée,T1,...,4.5,4.5,0.00,12.7,5,9.916667,0,5,0.0,216.0
2,333.0,1,1,ACA,YUL,J,ACA00876,2024-01-14 11:00:00,Arrivée,T1,...,0.0,0.0,0.00,7.0,1,11.000000,1,0,1.0,270.0
3,788.0,1,1,ACA,YUL,J,ACA00876,2024-05-11 09:55:00,Arrivée,T1,...,0.0,0.0,0.00,15.5,5,9.916667,1,0,2.0,255.0
4,NaN,1,3,AFR,NTE,J,AFR01489,2025-10-03 19:45:00,Arrivée,T1,...,0.0,0.0,0.00,4.8,10,19.750000,0,1,0.0,101.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
364740,NaN,4,3,SPC,LUX,D,SPC00195,2024-02-04 14:40:00,Départ,T1,...,0.0,0.0,0.00,8.7,2,14.666667,1,0,1.0,2.0
364741,NaN,4,3,SPC,OPF,D,SPC00364,2026-03-14 15:00:00,Départ,T1,...,13.1,11.8,0.91,21.7,3,15.000000,1,0,2.0,6.0
364742,339.0,1,3,CRL,ORY,J,CRL00753,2024-03-19 07:30:00,Départ,T1,...,0.0,0.0,0.00,11.0,3,7.500000,0,4,0.0,0.0
364743,NaN,4,3,VJH,CMF,D,VJH00373,2025-01-12 13:10:00,Départ,T1,...,0.0,0.0,0.00,27.2,1,13.166667,1,0,1.0,0.0


In [65]:
X_train = train_df[feature_cols].copy()
y_train = train_df[target_col].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df[target_col].copy()

In [66]:
RANDOM_STATE = 42

model = XGBRegressor(
    objective="reg:absoluteerror",
    eval_metric="mae",
    n_estimators=1623,
    learning_rate=0.06064746159809906,
    max_depth=13,
    min_child_weight=24,
    gamma=1.0767316383729433,
    subsample=0.8964027485905603,
    colsample_bytree=0.7548736579064788,
    reg_lambda=8.0,
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.7548736579064788, device=None,
             early_stopping_rounds=None, enable_categorical=False,
             eval_metric='mae', feature_types=None, feature_weights=None,
             gamma=1.0767316383729433, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.06064746159809906,
             max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=13, max_leaves=None,
             min_child_weight=24, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=1623, n_jobs=-1,
             num_parallel_tree=None, ...)

In [67]:
test_scored = test_df.copy()

pred_load_factor = model.predict(X_test)
pred_load_factor = np.clip(pred_load_factor, 0, 1.2)

test_scored["pred_load_factor"] = pred_load_factor

pred_pax = test_scored["pred_load_factor"] * test_scored["NbOfSeats"]
pred_pax = pred_pax.replace([np.inf, -np.inf], np.nan).fillna(0)

test_scored["pred_nbpaxtotal"] = np.round(
    np.clip(pred_pax, 0, None)
).astype(int)

test_scored["pred_nbpaxtotal"] = np.minimum(
    test_scored["pred_nbpaxtotal"],
    test_scored["NbOfSeats"]
).astype(int)

test_scored["abs_error"] = np.abs(test_scored["pred_nbpaxtotal"] - test_scored[TARGET])

test_scored[[
    "LTScheduledDatetime",
    "FlightNumberNormalized",
    "NbOfSeats",
    TARGET,
    "pred_nbpaxtotal",
    "abs_error"
]].head(20)

,LTScheduledDatetime,FlightNumberNormalized,NbOfSeats,NbPaxTotal,pred_nbpaxtotal,abs_error
0,2026-03-12 11:50:00,401,76.0,28.0,49,21.0
1,2026-03-12 19:50:00,47,19.0,17.0,13,4.0
2,2026-03-04 18:35:00,60,174.0,69.0,80,11.0
3,2026-03-07 14:20:00,230,186.0,108.0,150,42.0
4,2026-03-18 06:30:00,132,19.0,12.0,6,6.0
5,2026-03-20 12:40:00,299,189.0,181.0,147,34.0
6,2026-03-07 09:25:00,1088,180.0,134.0,150,16.0
7,2026-03-12 18:15:00,173,180.0,171.0,164,7.0
8,2026-03-09 19:40:00,194,76.0,57.0,56,1.0
9,2026-03-15 19:30:00,194,76.0,60.0,59,1.0


In [68]:
def regression_metrics(y_true, y_pred, name="dataset"):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    print(f"--- {name} ---")
    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R2   : {r2:.4f}")
    print()

regression_metrics(test_scored[TARGET], test_scored["pred_nbpaxtotal"], "Flight-level")

--- Flight-level ---
MAE  : 13.8623
RMSE : 23.5945
R2   : 0.8588

